# qLDPC benchmarks (`naive_n_dag`)

Runs `naive_n_dag` on every syndrome-extraction circuit in `inputs/qasm_files/qLDPC/`, once with each of its two configs in `benchmarks/configs/qLDPC/`:

- `<stem>.json`: FastSA placement
- `<stem>_deterministic.json`: deterministic placement (otherwise identical)

Schedules (and FastSA logs) go to `benchmarks/output/qLDPC/` as `<stem>.<placement>.naive_n_dag.schedule.txt`, the same names `benchmarks/run_naive_n_dag.py` uses. The execution time (`Texec`) and compile time (`Tcomp`) of every run are saved to `benchmarks/output/qLDPC/qldpc_timings.csv`, which is rewritten after each run, so an interrupted run keeps what finished.

These configs use the qLDPC physics (8 micrometer traps, 20000 m/s^2, 10 m/s), not the PowerMove settings of `analysis_notebook.ipynb`; the CSV records each run's lattice spacing and motion limits. The largest codes (up to 720 qubits) take the longest; set `PATTERN` below to run a subset.

In [ ]:
from pathlib import Path
import glob
import re
import sys
import time

import pandas as pd
from qiskit import QuantumCircuit

repo_root = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "src" / "naive_n_dag").is_dir())  # works from benchmarks/ or the repo root
src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from naive_n_dag.main import load_config, main as naive_n_dag_main

qasm_dir = repo_root / "inputs" / "qasm_files" / "qLDPC"
config_dir = repo_root / "benchmarks" / "configs" / "qLDPC"
output_dir = repo_root / "benchmarks" / "output" / "qLDPC"
output_dir.mkdir(parents=True, exist_ok=True)
timings_csv = output_dir / "qldpc_timings.csv"

final_time_re = re.compile(r"^final_time:\s*([0-9eE+\-.]+)\s+microsecond")


def parse_final_time_us(schedule_path: Path) -> float:
    for line in schedule_path.read_text(encoding="utf-8").splitlines():
        m = final_time_re.match(line.strip())
        if m:
            return float(m.group(1))
    raise ValueError(f"final_time not found in {schedule_path}")


def run_naive_n_dag(config_path: Path, qasm_path: Path, output_name: str, seed: int = 0, log: bool = False) -> dict:
    """Run naive_n_dag into output_dir; return the timing record."""
    config = load_config(config_path)
    t0 = time.perf_counter()
    exit_code = naive_n_dag_main(
        config,
        str(qasm_path),
        schedule_output_dir=output_dir,
        config_path=config_path,
        quiet=True,
        seed=seed,
        output_name=output_name,
        log=log,
    )
    wall_seconds = time.perf_counter() - t0
    schedule_path = output_dir / output_name
    return {
        "placement": config["fill_strategy"],
        "config": str(config_path.relative_to(repo_root)),
        "qasm": str(qasm_path.relative_to(repo_root)),
        "lattice_spacing": config["lattice_spacing"],
        "max_acceleration": config["max_acceleration"],
        "max_velocity": config["max_velocity"],
        "Tcomp (s)": wall_seconds,
        "Texec (µs)": parse_final_time_us(schedule_path),
        "exit_code": exit_code,
        "output_file": str(schedule_path.relative_to(repo_root)),
    }

## Circuits

`PATTERN` selects QASM files by name (`"*.qasm"` runs all of them, `"bb_n144*"` only bb144). They run smallest first.

In [ ]:
PATTERN = "*.qasm"

circuits = []
for path in sorted(qasm_dir.glob(PATTERN)):
    circuit = QuantumCircuit.from_qasm_file(str(path))
    two_qubit = sum(1 for i in circuit.data if len(i.qubits) == 2 and i.operation.name != "barrier")
    circuits.append({"code": path.stem, "qubits": circuit.num_qubits, "two-qubit gates": two_qubit, "qasm": path})
circuits.sort(key=lambda c: (c["qubits"], c["code"]))
pd.DataFrame(circuits).drop(columns="qasm")

## Run

Each circuit runs with its FastSA config and its deterministic config (missing configs are reported and skipped).

In [ ]:
records = []
for c in circuits:
    for placement, suffix in (("fastsa", ""), ("deterministic", "_deterministic")):
        config_path = config_dir / f"{c['code']}{suffix}.json"
        if not config_path.exists():
            print(f"SKIP {c['code']} [{placement}]: {config_path.relative_to(repo_root)} not found")
            continue
        rec = run_naive_n_dag(
            config_path,
            c["qasm"],
            f"{c['code']}.{placement}.naive_n_dag.schedule.txt",
            log=placement == "fastsa",
        )
        records.append({"code": c["code"], "qubits": c["qubits"], "two-qubit gates": c["two-qubit gates"], **rec})
        pd.DataFrame(records).to_csv(timings_csv, index=False)
        print(f"{c['code']} ({c['qubits']} qubits) [{placement}]: Texec={rec['Texec (µs)']:.1f} µs, Tcomp={rec['Tcomp (s)']:.1f} s")

print("timings:", timings_csv.relative_to(repo_root))

## Results

Execution time in milliseconds per code and placement, with the compile time in seconds.

In [ ]:
timings = pd.read_csv(timings_csv)
summary = timings.pivot_table(index=["code", "qubits", "two-qubit gates"], columns="placement",
                              values=["Texec (µs)", "Tcomp (s)"])
summary["Texec (µs)"] = summary["Texec (µs)"] / 1000
summary = summary.rename(columns={"Texec (µs)": "Texec (ms)"})[["Texec (ms)", "Tcomp (s)"]].sort_index(level="qubits")
summary.round(2)